In [3]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.model_selection import train_test_split

In [4]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\baselinedataset_trainsmall.pkl", 'rb') as f:
    dataset = pickle.load(f)

In [5]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\baselinedataset_test.pkl", 'rb') as f:
    test_dataset = pickle.load(f)

In [13]:
#Undersampling to meet class imbalances for class 0 and 1
X = dataset[:,:-1]
X_test = test_dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]
y_test = test_dataset[:, -1]
undersample = RandomUnderSampler(sampling_strategy='majority')
X, y = undersample.fit_resample(X, y)
X_test, y_test = undersample.fit_resample(X_test, y_test)

In [14]:
#Train test split
#X_train,X_test,y_train,y_test = train_test_split(X, y,test_size = 0.3, random_state = 42)
X_train = X
y_train = y
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  
    'eval_metric': 'logloss',        
    'max_depth': 3,                  
    'learning_rate': 0.1,            
    'subsample': 0.8,                
    'colsample_bytree': 0.8,         
    'seed': 42                       
}
num_rounds = 100  
model = xgb.train(params, dtrain, num_rounds)
y_pred = model.predict(dtest)
y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]  # Convert probabilities to binary predictions

print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))

Accuracy: 0.47221662975639217

Classification Report:
               precision    recall  f1-score   support

         0.0       0.48      0.58      0.52      4967
         1.0       0.46      0.36      0.41      4967

    accuracy                           0.47      9934
   macro avg       0.47      0.47      0.47      9934
weighted avg       0.47      0.47      0.47      9934



In [15]:
#Implementing Light GBM
import lightgbm as lgb
from sklearn.metrics import  roc_auc_score

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',  # for binary classification 
    'metric': 'auc', # area under the curve
    'boosting_type': 'gbdt',    # traditional Gradient Boosting Decision Tree
    'num_leaves': 31,           # number of leaves in one tree
    'learning_rate': 0.05,      # learning rate
    'feature_fraction': 0.9,    # feature fraction
    'bagging_fraction': 0.8,    # bagging fraction
    'bagging_freq': 5,          # bagging frequency
    'verbose': 0,               # 0 for silent mode
}
# Train the model
num_round = 100  # Number of boosting rounds
bst = lgb.train(params, train_data, num_round, valid_sets = [test_data])
# Make predictions on the test set
y_pred_prob_lgb = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_lgb = [1 if pred > 0.5 else 0 for pred in y_pred_prob_lgb]  # Convert probabilities to binary predictions

#Model evaluation 
accuracy = accuracy_score(y_test, y_pred_lgb)
roc_auc = roc_auc_score(y_test, y_pred_prob_lgb)

print(f'Accuracy on Test Set: {accuracy:.4f}')
print(f'ROC AUC on Test Set: {roc_auc:.4f}')
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.086417 seconds.
You can set `force_col_wise=true` to remove the overhead.
[1]	valid_0's auc: 0.528582
[2]	valid_0's auc: 0.521611
[3]	valid_0's auc: 0.533957
[4]	valid_0's auc: 0.542565
[5]	valid_0's auc: 0.523859
[6]	valid_0's auc: 0.516505
[7]	valid_0's auc: 0.513623
[8]	valid_0's auc: 0.508303
[9]	valid_0's auc: 0.506731
[10]	valid_0's auc: 0.506826
[11]	valid_0's auc: 0.507268
[12]	valid_0's auc: 0.504533
[13]	valid_0's auc: 0.499724
[14]	valid_0's auc: 0.495049
[15]	valid_0's auc: 0.498196
[16]	valid_0's auc: 0.500784
[17]	valid_0's auc: 0.501541
[18]	valid_0's auc: 0.496525
[19]	valid_0's auc: 0.496225
[20]	valid_0's auc: 0.496942
[21]	valid_0's auc: 0.493484
[22]	valid_0's auc: 0.491499
[23]	valid_0's auc: 0.48925
[24]	valid_0's auc: 0.485762
[25]	valid_0's auc: 0.485889
[26]	valid_0's auc: 0.490353
[27]	valid_0's auc: 0.48874
[28]	valid_0's auc: 0.488022
[29]	valid_0's auc: 0.489088
[30]	